## scRNA preprocessing

### Import libraries and setup files

In [ ]:
import sys
import decoupler as dc
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
sys.path.append('../utils')
from importing import download_h5ads, add_metadata_to_adata, build_adata, add_gene_names_to_adata
from plotting import plot_knee_curve, plot_expression_heatmap, stacked_barplot_proportions

import importlib
importlib.reload(sys.modules['importing'])
importlib.reload(sys.modules['plotting'])

### Create scanpy object and calculate basic QCs.

This assumes you have run the download RNA h5ads script. Here we create a scanpy object with no filtering and plot the RNA knee plot. 

Then we apply a very minimal filter of 10 UMIs per barcode and calculate some basic QCs per barcode to plot. 

In [ ]:
# download_and_extract_file(my_file)
h5ad_paths = [
"data/Subpool_1.h5ad",
"data/Subpool_2.h5ad",
"data/Subpool_3.h5ad",
"data/Subpool_4.h5ad",
"data/Subpool_5.h5ad"]

h5ad_map = download_h5ads(h5ad_paths)

adata = build_adata(h5ad_map)
adata.obs["AnalysisSet"] = adata.obs["sample"]
adata.obs["cell_barcode"] = adata.obs.index

print(adata.obs["sample"].value_counts())

#adata.obs["lab_sample_id"] = adata.obs["cell_barcode"].str.split('_', expand=True)[1]
add_gene_names_to_adata(adata, gene_metadata=gene_name_mapping)
adata.obs.index = adata.obs['cell_barcode']
adata.var['gene_id'] = adata.var.index
adata.var_names = adata.var['gene_name']
adata.var_names_make_unique()

#change gene_name column title with gene_name_unique
adata.var.rename(columns={'gene_name': 'gene_name_unique'}, inplace=True)

# Generate knee plot
plot_knee_curve(adata, output_folder="../results", prefix="plots")

In [ ]:
# QC scoring
# mitochondrial genes
adata.var["mt"] = adata.var_names.str.startswith("mt-")
# ribosomal genes
adata.var["ribo"] = adata.var_names.str.startswith(("Rps", "Rpl"))

#Keep barcodes with UMIs >=10
adata = adata[adata.X.sum(axis=1).A1 >= 10]

sc.pp.calculate_qc_metrics(
    adata, qc_vars=["mt", "ribo"], inplace=True, log1p=True
)

print(adata.obs["sample"].value_counts())

#adata = adata[adata.obs['total_counts'] < max_umi, :]
print("<150000 UMIs:", adata.shape, flush=True)

#adata = adata[adata.obs['n_genes_by_counts'] > min_genes, :] # min number of genes per cell
print(">250 genes:", adata.shape, flush=True)

#adata = adata[adata.obs['pct_counts_mt'] < max_mito, :]
print("<1% mt:", adata.shape, flush=True)

gc.collect()

In [ ]:
#This will be used in the joint plot scatterplot to calculate overlap with ATAC.
adata.obs.to_csv("results/unfiltered_rna_obs.tsv", sep = "\t")

In [ ]:
#violin plots

sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts"],
    groupby='sample',
    size=0,
    rotation = 90,
    multi_panel = True,
    show = True,
    log = True
)

In [ ]:
#violin plots

sc.pl.violin(
    adata,
    ["pct_counts_mt","pct_counts_ribo"],
    groupby='sample',
    size=0,
    rotation = 90,
    multi_panel = True,
    show = True,
    log = False
)

###  Now let's apply slightly more aggressive filtering and do basic clustering. 

After clustering, we can plot barplot showing Subpool proportions per leiden clusyer. We can bring in doublet scores calculated from preprocess_ATAC notebook to see which clusters are enriched in doublets, and validate by looking at total UMI counts per cluster.

In [ ]:
# cut-offs/parameters used:
min_umi = 500
max_umi = 150000
min_genes = 250
max_mito = 1
max_doublet = 0.1

knn_n_neighbors = 20
knn_n_pcs = 30

In [ ]:
adata = adata[adata.obs['total_counts'] >= min_umi, :]

adata = adata[adata.obs['total_counts'] < max_umi, :]
print("<150000 UMIs:", adata.shape, flush=True)

adata = adata[adata.obs['n_genes_by_counts'] > min_genes, :] 
print(">250 genes:", adata.shape, flush=True)

adata = adata[adata.obs['pct_counts_mt'] < max_mito, :]
print("<1% mt:", adata.shape, flush=True)

print(adata.obs["sample"].value_counts())

gc.collect()

In [ ]:
### normalize the data ###
sc.pp.normalize_total(adata, target_sum=1e4, layers=None, inplace=True) # Counts per 10k
sc.pp.log1p(adata, layer=None)
gc.collect()
# highly variable genes are used to compute the clustering 
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)

adatas = adata[:, adata.var.highly_variable]
gc.collect()
sc.pp.regress_out(adatas, ['pct_counts_mt','n_genes_by_counts'])
gc.collect()
sc.tl.pca(adatas, svd_solver='arpack')
gc.collect()

sc.pp.neighbors(adatas, n_neighbors=knn_n_neighbors, n_pcs=knn_n_pcs) # put non standard settings near the top
gc.collect()
print("Clustering....")
sc.tl.leiden(adatas, resolution = 1)
sc.tl.umap(adatas, random_state = 0)

adata.uns['neighbors'] = adatas.uns['neighbors']
adata.uns['leiden'] = adatas.uns['leiden']
adata.uns['umap'] = adatas.uns['umap']
adata.obs['leiden'] = adatas.obs['leiden']
adata.obsm = adatas.obsm
adata.obsp = adatas.obsp
gc.collect()

sc.pl.umap(adata, 
           color=['leiden'], 
           size=10, 
           legend_loc = 'on data', 
           legend_fontsize=15,
           show = False
          )

In [ ]:
#Subpool distribution by leiden

stacked_barplot_proportions(adata.obs, cluster_key="leiden", var_key="sample", reverse_order=True)

In [ ]:
filt_atac_obs = pd.read_csv("results/filt-atac-obs.tsv", sep = "\t")
filt_atac_obs["barcode"] = [s.split(":")[1] for s in atac_adata.obs_names]
filt_atac_obs["is_doublet"] = "False"
filt_atac_obs.loc[filt_atac_obs["doublet_probability"] >=0.7 ,"is_doublet"] = "True"
filt_atac_obs

In [ ]:
#load ATAC observations to compare 

filt_rna_obs = adata.obs
filt_rna_obs_indexed = filt_rna_obs.set_index("cell_barcode")
filt_atac_obs_indexed = filt_atac_obs.set_index("barcode")
common_barcodes = list(set(filt_rna_obs_indexed.index) & set(filt_atac_obs_indexed.index))

filt_rna_obs_indexed["is_doublet"] = "Undetermined"
filt_rna_obs_indexed.loc[ common_barcodes, "is_doublet"] = filt_atac_obs_indexed.loc[common_barcodes, "is_doublet"]

In [ ]:
#Doublet assignment per cluster
stacked_barplot_proportions(filt_rna_obs_indexed, cluster_key="leiden", var_key="is_doublet", reverse_order=True)


In [ ]:
#violin plot of total counts by leiden
sc.pl.violin(adata, groupby="leiden", keys = "total_counts", stripplot=False)

In [ ]:
adata.write_h5ad("results/filtered-RNA.h5ad")